In [ ]:
import os
import gc

import numpy as np
import tifffile
import scipy.ndimage as ndi
import cupy as cp
import cupyx.scipy.ndimage as cpx_ndi

In [ ]:
def make_psf(size, sigma):
    """Create a normalized 3D Gaussian PSF."""
    p = np.zeros((size, size, size))
    p[size // 2, size // 2, size // 2] = 1
    p = ndi.gaussian_filter(p, sigma)
    p /= p.sum()
    return p


def normalize_and_convert_to_float(array):
    """Normalize values to [0, 1] using min–max scaling."""
    float_array = array.astype(np.float32)
    min_val = float_array.min()
    max_val = float_array.max()
    normalized_array = (float_array - min_val) / (max_val - min_val)
    return normalized_array


def LR_deconv(img, psf, i):
    """Perform Richardson–Lucy deconvolution (CPU)."""
    im = img.copy()
    for n in range(i):
        print(n)
        normalize_and_convert_to_float(im)
        im1 = im.copy()
        conv1 = img / ndi.convolve(im1, psf, mode="constant", cval=0.0)
        conv2 = ndi.convolve(conv1, psf, mode="constant", cval=0.0)
        im = im1 * conv2
    return im


def LR_deconv_GPU(img, psf, i, zval=40, overlap=10, zoom=(2.5, 2.5, 2.5)):
    """Perform Richardson–Lucy deconvolution (GPU)."""
    psf = normalize_and_convert_to_float(psf)
    psf_cp = cp.asarray(psf).astype(cp.float32)
    im = img.astype(cp.float32)
    result = np.empty_like(im)

    for start_z in range(0, im.shape[0], zval - overlap):
        print(f"\tProcess deconvolution:{start_z}")
        end_z = min(start_z + zval, im.shape[0])

        im_chunk = im[start_z:end_z]
        im_chunk_gpu = cp.asarray(im_chunk).astype(cp.float32)
        im_chunk_gpu_d = im_chunk_gpu.copy()

        for n in range(i):
            conv1 = cp.divide(
                im_chunk_gpu_d,
                cpx_ndi.convolve(im_chunk_gpu, psf_cp, mode="constant", cval=0.0),
            )
            conv2 = cpx_ndi.convolve(conv1, psf_cp, mode="constant", cval=0.0)
            if n != (i - 1):
                im_chunk_gpu *= conv2

        if start_z == 0:
            result[start_z:end_z] = cp.asnumpy(im_chunk_gpu)
        else:
            result[start_z + overlap // 2 : end_z] = cp.asnumpy(im_chunk_gpu[overlap // 2 :])

        cp.get_default_memory_pool().free_all_blocks()

    cp.get_default_memory_pool().free_all_blocks()
    return result


def read_tiff_stack(folder_path, start_index=0, num_images=-1):
    """Load a TIFF stack from a folder containing sequential TIFF files."""
    tiff_files = [
        os.path.join(folder_path, fn)
        for fn in os.listdir(folder_path)
        if fn.lower().endswith(('.tif', '.tiff'))
    ]
    tiff_files.sort()

    if num_images == -1:
        tiff_files = tiff_files[start_index:]
    else:
        tiff_files = tiff_files[start_index:(start_index + num_images)]

    print(f"Load tiff: {start_index}-{len(tiff_files)}")

    first = tifffile.imread(tiff_files[0])
    stack = np.empty((len(tiff_files),) + first.shape, dtype=np.uint16)

    for i, tf in enumerate(tiff_files):
        stack[i] = tifffile.imread(tf)

    return stack

In [ ]:
src = "/path/to/source_dir"
dst = "/path/to/output_dir"
pre_img_path = os.path.join(src, "circadian_1st", "circadian_1st_Reconst")
deconv_path = os.path.join(src, "circadian_1st", "circadian_1st_Deconv")
os.makedirs(deconv_path, exist_ok=True)

In [ ]:
# Deconvolution

color_name = "cFos"  # Input directory name for the 1st color

psf = make_psf(9, (1.8, 1.8, 1.8))
z_chunk = 100

c = 0
for dir in os.listdir(pre_img_path):
    if "CT0" in dir:
        continue
    if "Reconst" in dir:
        c += 1
        if c < 2:
            continue
        print(dir)
        for dir2 in os.listdir(os.path.join(pre_img_path, dir)):
            if "cfos" in dir2:
                FPr = os.path.join(pre_img_path, dir, dir2)
                FPw = os.path.join(deconv_path, dir, dir2)
                if os.path.exists(FPw):
                    continue
                os.makedirs(FPw, exist_ok=True)

                zoom_factors = (2.5, 2.5, 2.5)
                overlap = psf.shape[0] + 7
                overlap_zoom = int(psf.shape[0] * 2)  # Overlap adjusted for zoom
                im = read_tiff_stack(FPr)
                im[im < 1000] = 1000
                os.makedirs(FPw, exist_ok=True)

                for start_z in range(0, im.shape[0], z_chunk - overlap // 2):
                    print(f"Process z:{start_z}")
                    end_z = min(start_z + z_chunk, im.shape[0])

                    im2 = cp.asnumpy(
                        cpx_ndi.zoom(
                            cp.asarray(im[start_z:end_z] // 2),
                            zoom=zoom_factors,
                            order=3,
                        )
                    )
                    cp.get_default_memory_pool().free_all_blocks()

                    RC = LR_deconv_GPU(im2, psf, 10, zval=40, overlap=overlap_zoom)
                    del im2
                    gc.collect()
                    cp.get_default_memory_pool().free_all_blocks()

                    RC = np.clip(RC * 2, 0, 65535).astype(np.uint16)

                    for i, z_slice in enumerate(RC):
                        if start_z != 0 and i < overlap // 2:
                            continue
                        filename = f"{int(start_z * zoom_factors[0] + i):08d}.tif"
                        tifffile.imwrite(os.path.join(FPw, filename), z_slice)

                    del RC
                    gc.collect()

                del im
                gc.collect()